# NUS CS/AI Curriculum Analytics
**Môn học:** Nhập môn Khoa học Dữ liệu — Học viện Công nghệ Bưu chính Viễn thông (PTIT)  
**Nguồn dữ liệu:** [NUSMods API v2](https://api.nusmods.com/v2/) (Dữ liệu mở 5 năm: 2022-2023 -> 2026-2027)  
**Quy chuẩn thiết kế:** Tối đa hóa tỷ lệ Data-Ink của **Edward Tufte** và Nghệ thuật Kể chuyện qua Dữ liệu của **Cole Nussbaumer Knaflic (Storytelling with Data)**.

---


## 1. Khởi tạo môi trường và thiết lập phong cách trực quan
Nạp các thư viện tính toán số liệu, học máy và cấu hình font Segoe UI theo bảng màu tối giản: Navy (`#1A365D`), Crimson (`#C53030`), Slate Gray (`#718096`).


In [ ]:
import ast
import json
import re
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud

import sys
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from loader import load_or_fetch, build_dataframe, deduplicate_latest_modules

# Hằng số màu sắc chuẩn Tufte & Storytelling with Data
COLOR_PRIMARY = "#1A365D"      # Xanh Navy cho tín hiệu trọng tâm
COLOR_ACCENT = "#C53030"       # Đỏ Crimson cho môn AI và điểm nhấn
COLOR_MUTED = "#718096"        # Xám tro cho thông tin nền/bối cảnh
COLOR_LIGHT = "#E2E8F0"        # Xám nhạt cho lưới tham chiếu mờ
COLOR_DARK = "#2D3748"         # Màu chữ than chì
COLOR_BG = "#FFFFFF"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "text.color": COLOR_DARK,
    "axes.labelcolor": COLOR_DARK,
    "axes.edgecolor": COLOR_LIGHT,
    "axes.linewidth": 0.8,
    "xtick.color": COLOR_MUTED,
    "ytick.color": COLOR_MUTED,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "figure.titlesize": 15,
    "figure.titleweight": "bold",
    "grid.color": COLOR_LIGHT,
    "grid.linestyle": ":",
    "grid.linewidth": 0.6,
    "figure.facecolor": COLOR_BG,
    "axes.facecolor": COLOR_BG,
})

def tufte_spines(ax):
    """Loại bỏ viền trên/phải, làm mờ viền trái/dưới theo nguyên tắc Tufte."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(COLOR_MUTED)
    ax.spines["bottom"].set_color(COLOR_MUTED)
    ax.tick_params(direction="out", length=4, width=0.8, colors=COLOR_MUTED)

print("Khởi tạo môi trường thành công.")


## 2. Block 1: thu thập dữ liệu
Tải toàn bộ học phần CS/IS/DSA từ NUSMods API trong 5 năm học gần nhất (2022-2023 đến 2026-2027) với cơ chế lưu cache file JSON để bảo đảm tính tái lập nhanh chóng.


In [ ]:
raw_data = load_or_fetch(cache=ROOT_DIR / "data" / "raw_nus_cs.json")
for yr, mods in raw_data.items():
    print(f"Năm học {yr}: Đã thu thập {len(mods)} học phần.")


## 3. Block 2: tiền xử lý, trích xuất đặc trưng và chuẩn hóa
- Phân rã mảng `workload` thành 5 cột số giờ và tính tổng `total_workload`.
- Trích xuất cấp độ học phần `level` chuẩn hóa thang số nguyên từ 1 đến 6 (Level 1 - Level 6).
- Điền khuyết thiếu bằng giá trị trung vị (`median imputation`) theo từng cấp độ `level`.
- Bóc tách đệ quy cây tiên quyết `prereqTree` thành mảng mã môn `prereq_codes` và số lượng `prereq_count`.
- Tạo nhãn phân tầng `study_tier` (`Đại học` cho Level 1 - 4, `Sau đại học` cho Level 5 - 6) và nhãn `is_ai_core`.
- Chuẩn hóa thang đo Min-Max Scaling [0, 1].


In [ ]:
df = build_dataframe(raw_data)
df.to_csv(str(ROOT_DIR / "data" / "cleaned_nus_cs.csv"), index=False, encoding="utf-8")
df_unique = deduplicate_latest_modules(df)
df_unique.to_csv(str(ROOT_DIR / "data" / "unique_nus_cs.csv"), index=False, encoding="utf-8")
print(f"Kích thước bảng dữ liệu sạch: {df.shape}")
print(f"Kích thước bảng môn học duy nhất: {df_unique.shape}")
df[["moduleCode", "title", "moduleCredit", "level", "total_workload", "prep_hours", "prereq_count", "is_ai_core", "study_tier"]].head(5)


## 4. Block 3: trực quan hóa dữ liệu (Tufte & Storytelling with Data)
### Biểu đồ 1: Boxplot Phân phối Giờ Tự học theo Cấp độ Học phần (Level 1 - 6)
- Tối đa hóa tỷ lệ Data-Ink: Loại bỏ viền khung hộp rườm rà.
- Nhãn trực tiếp (Direct Labeling): Ghi rõ nhãn trung vị `Trung vị: Xh` ngay cạnh đường trung vị (không có dấu `=`).


In [ ]:
df_unique = df.sort_values("academicYear").drop_duplicates(subset=["moduleCode"], keep="last").copy()

fig, ax = plt.subplots(figsize=(10, 6))
palette = {"Đại học": "#2B6CB0", "Sau Đại học": "#C53030"}

sns.boxplot(
    data=df_unique,
    x="level",
    y="prep_hours",
    hue="study_tier",
    palette=palette,
    width=0.5,
    fliersize=3.5,
    linewidth=1.1,
    boxprops=dict(alpha=0.85),
    ax=ax
)

medians = df_unique.groupby("level")["prep_hours"].median()
for i, (lvl, med) in enumerate(medians.items()):
    ax.text(
        i + 0.28, med, f" Trung vị: {med:.0f}h",
        ha="left", va="center", fontsize=9, fontweight="bold", color=COLOR_PRIMARY,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="#FFFFFF", edgecolor="#CBD5E0", alpha=0.95, lw=0.8)
    )

ax.set_title(
    "Nhu cầu tự học tăng từ 3.0h lên 4.0h/tuần ở bậc Sau Đại học (Level 5 - 6)",
    loc="left", pad=28, fontsize=13.5, fontweight="bold", color=COLOR_PRIMARY
)
ax.text(
    0, 1.04,
    "Phân phối số giờ tự nghiên cứu chuẩn bị (prep_hours/tuần) theo từng cấp độ học phần (Level 1 đến 6). Nhãn: Trung vị (Median).",
    transform=ax.transAxes, fontsize=10, color=COLOR_MUTED
)

ax.set_xlabel("Cấp độ môn học (Level 1 - 6)", fontsize=11, labelpad=8)
ax.set_ylabel("Số giờ tự học hàng tuần (Giờ chuẩn bị / tuần)", fontsize=11, labelpad=8)
ax.set_ylim(-0.5, 9.5)
ax.grid(axis="y", linestyle=":", alpha=0.6)
tufte_spines(ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Bậc Đại học (Level 1 - 4)", "Bậc Sau Đại học (Level 5 - 6)"],
          frameon=False, loc="upper left", fontsize=10)

plt.tight_layout()
plt.show()


### Biểu đồ 2: Xu hướng Môn học Mở & Tỷ trọng Môn AI qua 5 Năm học (Tương tác Plotly)
Đường xám trung tính thể hiện tổng thể chương trình đào tạo, đường đỏ Crimson nhấn mạnh sự phát triển của các môn Trí tuệ Nhân tạo.


In [ ]:
yearly_stats = []
years = sorted(df["academicYear"].unique())
for yr in years:
    sub = df[df["academicYear"] == yr]
    yearly_stats.append({"year": yr, "total": len(sub), "ai": sub["is_ai_core"].sum()})

df_line_summary = pd.DataFrame(yearly_stats)

fig_line = go.Figure()

fig_line.add_trace(go.Scatter(
    x=df_line_summary["year"],
    y=df_line_summary["total"],
    mode="lines+markers+text",
    name="Tổng số môn CS/IS/DSA",
    line=dict(color=COLOR_MUTED, width=2.5),
    marker=dict(size=7, color=COLOR_MUTED),
    text=df_line_summary["total"],
    textposition="top center",
    textfont=dict(size=11, color=COLOR_MUTED),
    hovertemplate="Năm học: <b>%{x}</b><br>Tổng số môn học: <b>%{y} môn</b><extra></extra>"
))

fig_line.add_trace(go.Scatter(
    x=df_line_summary["year"],
    y=df_line_summary["ai"],
    mode="lines+markers+text",
    name="Môn trọng tâm AI/ML",
    line=dict(color=COLOR_ACCENT, width=3.2),
    marker=dict(size=9, color=COLOR_ACCENT),
    text=df_line_summary["ai"],
    textposition="bottom center",
    textfont=dict(size=11, color=COLOR_ACCENT),
    hovertemplate="Năm học: <b>%{x}</b><br>Môn Trí tuệ Nhân tạo: <b>%{y} môn</b><extra></extra>"
))

fig_line.update_layout(
    title=dict(
        text="<b>Môn học Trí tuệ Nhân tạo tăng trưởng bền bỉ trong 5 năm học gần nhất</b><br>"
             "<span style='font-size:12px; color:#718096;'>Số lượng môn học CS mở mới tăng từ 271 lên 296 môn, trong đó AI chiếm ~30% toàn bộ chương trình</span>",
        x=0.04, y=0.92
    ),
    plot_bgcolor=COLOR_BG,
    paper_bgcolor=COLOR_BG,
    font=dict(family="Segoe UI, Arial, sans-serif", color=COLOR_DARK),
    xaxis=dict(showgrid=False, linecolor=COLOR_MUTED, tickfont=dict(size=11)),
    yaxis=dict(showgrid=True, gridcolor=COLOR_LIGHT, linecolor="rgba(0,0,0,0)", title="Số môn học mở"),
    legend=dict(x=0.04, y=0.75, bgcolor="rgba(255,255,255,0.7)", bordercolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=40, t=90, b=50)
)
fig_line.show()


### Biểu đồ 3: Tương quan Cấp độ Môn học và Giờ Tự học kèm Hồi quy OLS (Tương tác Plotly)
Trục X được mã hóa từ Level 1 đến Level 6. Toàn bộ tooltip được Việt hóa hoàn toàn và triệt tiêu các nhãn mặc định có dấu `=`.


In [ ]:
fig_scatter = go.Figure()

sub_std = df_unique[~df_unique["is_ai_core"]]
cust_std = np.stack((sub_std["moduleCode"], sub_std["title"], sub_std["level"], sub_std["prep_hours"]), axis=-1)
fig_scatter.add_trace(go.Scatter(
    x=sub_std["level"],
    y=sub_std["prep_hours"],
    mode="markers",
    name="Môn CS Tiêu chuẩn",
    marker=dict(color="#A0AEC0", size=7),
    opacity=0.65,
    customdata=cust_std,
    hovertemplate="<b>%{customdata[1]}</b><br>"
                  "Mã môn học: <b>%{customdata[0]}</b><br>"
                  "Cấp độ: <b>Level %{customdata[2]}</b><br>"
                  "Giờ tự học: <b>%{customdata[3]:.1f} giờ/tuần</b><br>"
                  "Phân loại: Môn CS Tiêu chuẩn<extra></extra>"
))

sub_ai = df_unique[df_unique["is_ai_core"]]
cust_ai = np.stack((sub_ai["moduleCode"], sub_ai["title"], sub_ai["level"], sub_ai["prep_hours"]), axis=-1)
fig_scatter.add_trace(go.Scatter(
    x=sub_ai["level"],
    y=sub_ai["prep_hours"],
    mode="markers",
    name="Môn Trọng tâm AI/ML",
    marker=dict(color=COLOR_ACCENT, size=8),
    opacity=0.8,
    customdata=cust_ai,
    hovertemplate="<b>%{customdata[1]}</b><br>"
                  "Mã môn học: <b>%{customdata[0]}</b><br>"
                  "Cấp độ: <b>Level %{customdata[2]}</b><br>"
                  "Giờ tự học: <b>%{customdata[3]:.1f} giờ/tuần</b><br>"
                  "Phân loại: Môn Trọng tâm AI/ML<extra></extra>"
))

for name, sub, col, dash in [("Tất cả môn học", df_unique, COLOR_PRIMARY, "solid"), ("Môn AI/ML", sub_ai, COLOR_ACCENT, "dash")]:
    res = stats.linregress(sub["level"], sub["prep_hours"])
    x_vals = np.array([1, 6])
    y_vals = res.slope * x_vals + res.intercept
    fig_scatter.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines",
        name=f"Xu hướng OLS ({name})",
        line=dict(color=col, width=2.2, dash=dash),
        hovertemplate=f"Đường xu hướng OLS ({name}):<br>"
                      f"Độ dốc: {res.slope:+.2f} giờ/cấp độ (p = {res.pvalue:.2e})<extra></extra>"
    ))

fig_scatter.update_layout(
    title=dict(
        text="<b>Cấp độ môn học càng cao, sinh viên càng phải dành nhiều giờ tự học</b><br>"
             "<span style='font-size:12px; color:#718096;'>Đường xu hướng OLS (p < 0.001) cho thấy độ dốc đi lên rõ rệt; môn AI (đỏ) có mật độ tự học trải dài từ 2 đến 8 giờ/tuần.</span>",
        x=0.04, y=0.93
    ),
    plot_bgcolor=COLOR_BG,
    paper_bgcolor=COLOR_BG,
    font=dict(family="Segoe UI, Arial, sans-serif", color=COLOR_DARK),
    xaxis=dict(showgrid=False, linecolor=COLOR_MUTED, dtick=1, range=[0.5, 6.5], title="Cấp độ môn học (Level 1 - 6)"),
    yaxis=dict(showgrid=True, gridcolor=COLOR_LIGHT, linecolor="rgba(0,0,0,0)", title="Số giờ tự học hàng tuần (Giờ/tuần)"),
    legend=dict(x=0.04, y=0.82, bgcolor="rgba(255,255,255,0.85)", bordercolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=40, t=85, b=50)
)
fig_scatter.show()


### Biểu đồ 4: Ma trận Tương quan Nửa Tam giác Dưới (Tufte Half-Matrix)
Loại bỏ 50% diện tích đối xứng dư thừa, triệt tiêu hoàn toàn các hàng/cột trống (dead ink) để đảm bảo mọi ô vuông đều truyền tải thông tin đắt giá.


In [ ]:
corr_cols = ["moduleCredit", "lec_hours", "tut_hours", "lab_hours", "proj_hours", "prep_hours", "prereq_count"]
label_names = ["Số tín chỉ", "Giờ giảng bài", "Giờ thảo luận", "Giờ thực hành", "Giờ đồ án", "Giờ tự học", "Môn tiên quyết"]
corr_mat = df_unique[corr_cols].corr()
corr_mat.columns = label_names
corr_mat.index = label_names

corr_sub = corr_mat.iloc[1:, :-1]
mask_sub = np.triu(np.ones_like(corr_sub, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(
    corr_sub,
    mask=mask_sub,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-0.6,
    vmax=0.6,
    cbar_kws={"shrink": 0.75, "label": "Hệ số tương quan Pearson (r)"},
    linewidths=0.8,
    linecolor=COLOR_BG,
    ax=ax
)

plt.xticks(rotation=30, ha="right", fontsize=9.5)
plt.yticks(rotation=0, fontsize=9.5)

ax.set_title(
    "Tương quan các yếu tố khối lượng học tập và điều kiện tiên quyết",
    loc="left", pad=25, fontsize=13, fontweight="bold", color=COLOR_PRIMARY
)
ax.text(
    0, 1.03,
    "Nửa ma trận tam giác dưới: Số tín chỉ tương quan mạnh nhất với Giờ đồ án (0.59) và Giờ thảo luận (0.35).",
    transform=ax.transAxes, fontsize=10, color=COLOR_MUTED
)

plt.tight_layout()
plt.show()


### Biểu đồ 5: Phân tầng Cơ cấu đào tạo theo Mã ngành, Bậc học & Cấp độ (Interactive Treemap)
Hỗ trợ drill-down phân cấp 4 tầng. Định dạng `hovertemplate` tiếng Việt thuần túy, loại bỏ hoàn toàn các trường mặc định có dấu `=` (`labels=`, `parent=`, `id=`).


In [ ]:
df_unique["prefix"] = df_unique["moduleCode"].str.extract(r"^([A-Z]{2,3})")
dept_map = {"CS": "Khoa Khoa học Máy tính (CS)", "IS": "Khoa Hệ thống Thông tin (IS)", "DSA": "Khoa Khoa học Dữ liệu (DSA)"}
df_unique["dept_label"] = df_unique["prefix"].map(dept_map).fillna(df_unique["prefix"])
df_unique["tier_label"] = df_unique["study_tier"].map({"Đại học": "Bậc Đại học", "Sau Đại học": "Bậc Sau Đại học"}).fillna(df_unique["study_tier"])
df_unique["level_label"] = "Cấp độ " + df_unique["level"].astype(str)
df_unique["module_display"] = df_unique["moduleCode"] + ": " + df_unique["title"]

fig_treemap = px.treemap(
    df_unique,
    path=["dept_label", "tier_label", "level_label", "module_display"],
    values="moduleCredit",
    color="total_workload",
    color_continuous_scale="Blues"
)

fig_treemap.update_traces(
    hovertemplate="<b>%{label}</b><br>"
                  "Trực thuộc: <b>%{parent}</b><br>"
                  "Số tín chỉ: <b>%{value}</b><br>"
                  "Khối lượng học tập: <b>%{color:.1f} giờ/tuần</b><extra></extra>"
)

fig_treemap.update_layout(
    title=dict(
        text="<b>Cấu trúc Chương trình Đào tạo theo Mã ngành, Bậc học và Cấp độ</b><br>"
             "<span style='font-size:12px; color:#718096;'>Độ lớn ô tương ứng số tín chỉ; độ đậm màu xanh thể hiện khối lượng học tập (Giờ học/tuần). Nhấp vào từng ô để phóng to xem chi tiết.</span>",
        x=0.03, y=0.94
    ),
    paper_bgcolor=COLOR_BG,
    font=dict(family="Segoe UI, Arial, sans-serif", color=COLOR_DARK),
    margin=dict(l=30, r=30, t=85, b=30)
)
fig_treemap.show()


### Biểu đồ 6: Xếp hạng Tòa nhà & Giảng đường Tổ chức Môn CS/AI tại Khuôn viên NUS
Biểu đồ thanh ngang (Horizontal Bar Chart) giúp người đọc không phải nghiêng đầu để đọc tên các tòa nhà dài.


In [ ]:
venue_counter = {}
for y, mods in raw_data.items():
    for m in mods:
        for s in m.get("semesterData", []):
            for t in s.get("timetable", []):
                v = t.get("venue")
                if v and isinstance(v, str) and v.upper() not in ["ONLINE", "E-LEARN"]:
                    b_match = re.match(r"^([A-Z]+[0-9]*[A-Z]*)", v)
                    bldg = b_match.group(1) if b_match else v
                    venue_counter[bldg] = venue_counter.get(bldg, 0) + 1

df_venues = pd.DataFrame(
    sorted(venue_counter.items(), key=lambda x: x[1], reverse=True)[:15],
    columns=["venue", "slots"]
).sort_values("slots", ascending=True)

colors = [COLOR_PRIMARY if v in ["COM1", "COM3"] else COLOR_MUTED for v in df_venues["venue"]]

fig_venues = go.Figure(go.Bar(
    x=df_venues["slots"],
    y=df_venues["venue"],
    orientation="h",
    marker=dict(color=colors),
    text=df_venues["slots"],
    textposition="outside",
    textfont=dict(size=11, color=COLOR_DARK),
    hovertemplate="Tòa nhà giảng đường: <b>%{y}</b><br>Tổng số tiết học: <b>%{x:,} tiết</b><extra></extra>"
))

fig_venues.update_layout(
    title=dict(
        text="<b>COM1 và COM3 là hai trung tâm giảng dạy lớn nhất của Khoa Máy tính NUS</b><br>"
             "<span style='font-size:12px; color:#718096;'>Số tiết học (slots) được tổ chức tại các tòa nhà/giảng đường khuôn viên NUS trong 5 năm</span>",
        x=0.04, y=0.92
    ),
    plot_bgcolor=COLOR_BG,
    paper_bgcolor=COLOR_BG,
    font=dict(family="Segoe UI, Arial, sans-serif", color=COLOR_DARK),
    xaxis=dict(showgrid=True, gridcolor=COLOR_LIGHT, linecolor="rgba(0,0,0,0)", range=[0, 4800], title="Tổng số tiết học"),
    yaxis=dict(showgrid=False, linecolor=COLOR_MUTED),
    margin=dict(l=80, r=60, t=90, b=50)
)
fig_venues.show()


### Biểu đồ 7a & 7b: Từ khóa Học thuật AI (WordCloud) & Chuỗi Tiên quyết AI (Directed Acyclic Graph)
- **WordCloud:** Làm sạch toàn diện các từ dừng học thuật rườm rà.
- **Network DAG:** Luồng đào tạo 4 giai đoạn rõ ràng từ Toán nền tảng đến Chuyên sâu AI.


In [ ]:
# 7a. WordCloud
raw_ai_text = " ".join(df_unique[df_unique["is_ai_core"]]["description"].fillna("").astype(str))
stopwords_extended = {
    "module", "course", "courses", "students", "student", "will", "use", "using", "used", "study",
    "studies", "topics", "topic", "provide", "provides", "including", "include", "includes", "also",
    "various", "techniques", "technique", "methods", "method", "one", "two", "three", "based", "well", "such",
    "and", "to", "the", "in", "of", "for", "with", "a", "an", "is", "are",
    "this", "that", "as", "by", "on", "from", "at", "be", "it", "or", "through",
    "can", "their", "have", "has", "do", "does", "how", "what", "which",
    "application", "applications", "apply", "project", "projects", "problem", "problems", "skills", "knowledge",
    "introduce", "introduces", "introduction", "aims", "cover", "covers", "covered", "basic", "advanced",
    "practical", "principles", "principle", "technologies", "technology", "concept", "concepts", "issue", "issues",
    "design", "development", "develop", "develops", "solution", "solutions", "hands", "gain", "understanding",
    "understand", "system", "systems", "computational", "programming", "program", "management", "integration",
    "area", "areas", "example", "examples", "process", "processes", "build", "building", "work", "working",
    "tool", "tools", "these", "its", "human", "explore", "explores", "part", "parts", "domain", "domains",
    "fundamental", "fundamentals", "real", "world", "key", "modern", "solving", "solve", "theory", "theoretical",
    "approaches", "approach", "field", "fields", "require", "requires", "required", "prerequisite", "prerequisites",
    "taught", "teaching", "learn"
}
tokens = re.findall(r"\b[a-zA-Z]{3,}\b", raw_ai_text.lower())
cleaned_words = [t for t in tokens if t not in stopwords_extended]
cleaned_text = " ".join(cleaned_words)

def clean_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    w = word.lower()
    if w in {"neural", "deep", "learning", "reinforcement", "vision", "nlp", "intelligence", "robotics", "transformer", "generative"}:
        return COLOR_ACCENT
    elif w in {"data", "model", "models", "algorithm", "algorithms", "optimization", "decision", "network", "networks", "inference", "reasoning", "classification", "probabilistic"}:
        return COLOR_PRIMARY
    return COLOR_MUTED

wc = WordCloud(width=1200, height=600, background_color="white", color_func=clean_color_func, font_step=2, max_words=70, random_state=42).generate(cleaned_text)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Từ khóa học thuật cốt lõi trong các học phần AI/ML tại Đại học Quốc gia Singapore", loc="left", pad=15, fontsize=13, fontweight="bold", color=COLOR_PRIMARY)
plt.tight_layout()
plt.show()

# 7b. Tạo biểu đồ DAG tương tác động HTML
from utils import generate_interactive_dag_html
dag_path = generate_interactive_dag_html('output/07b_network_prereq.html')
print(f'Đã tạo biểu đồ DAG tương tác HTML tại: {dag_path}')
try:
    from IPython.display import IFrame, display
    display(IFrame(src='output/07b_network_prereq.html', width='100%', height=650))
except Exception:
    pass



## 5. thống kê suy luận và phân cụm máy học (Statistical Inference & ML)
### 5.1. Kiểm định Giả thuyết Thống kê (Welch's Two-Sample t-test) & Khoảng Tin cậy 95%
- Giả thuyết $H_0: \mu_{\text{Đại học}} = \mu_{\text{Sau đại học}}$.
- Giả thuyết $H_1: \mu_{\text{Đại học}} \neq \mu_{\text{Sau đại học}}$.


In [ ]:
ug_prep = df[df["study_tier"] == "Đại học"]["prep_hours"]
pg_prep = df[df["study_tier"] == "Sau Đại học"]["prep_hours"]
t_stat, p_val = stats.ttest_ind(ug_prep, pg_prep, equal_var=False)

ci_95 = stats.t.interval(
    0.95,
    df=len(df) - 1,
    loc=df["total_workload"].mean(),
    scale=stats.sem(df["total_workload"])
)

print(f"Khối lượng tự học trung bình Bậc Đại học (Level 1 - 4)    : {ug_prep.mean():.2f} giờ/tuần (N={len(ug_prep)})")
print(f"Khối lượng tự học trung bình Bậc Sau Đại học (Level 5 - 6) : {pg_prep.mean():.2f} giờ/tuần (N={len(pg_prep)})")
print(f"Kết quả kiểm định Welch t-test : t = {t_stat:.4f}, p-value = {p_val:.4e}")
if p_val < 0.05:
    print("=> Kết luận: Bác bỏ giả thuyết H0 (p < 0.05). Khối lượng tự học bậc Sau Đại học cao hơn rõ rệt.")
else:
    print("=> Kết luận: Chưa đủ bằng chứng bác bỏ H0.")

print(f"Ước lượng Khoảng tin cậy 95% cho tổng số giờ học: [{ci_95[0]:.2f} giờ/tuần, {ci_95[1]:.2f} giờ/tuần]")


### 5.2. Phân cụm Chủ đề Môn học Không giám sát (TF-IDF + PCA 2D + K-Means)
Phương pháp Elbow Method xác định điểm gãy tối ưu tại $K = 3$, tự động phân tách 3 chủ đề học thuật chính.


In [ ]:
texts = df_unique["description"].fillna("").tolist()
tfidf = TfidfVectorizer(max_features=500, stop_words="english")
X = tfidf.fit_transform(texts)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X.toarray())

km_final = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_pca)
cluster_map = {
    0: "Chủ đề 1: Trí tuệ Nhân tạo & Xử lý Dữ liệu",
    1: "Chủ đề 2: Kỹ thuật Phần mềm & Hệ thống Máy tính",
    2: "Chủ đề 3: Lý thuyết Tính toán & Thuật toán Nền tảng"
}
df_unique["cluster_label"] = [cluster_map.get(c, f"Cụm {c+1}") for c in km_final.labels_]
df_unique["pca_1"] = X_pca[:, 0]
df_unique["pca_2"] = X_pca[:, 1]

fig_clusters = go.Figure()
colors_list = ["#1A365D", "#2B6CB0", "#C53030"]

for idx, (c_id, c_name) in enumerate(cluster_map.items()):
    sub = df_unique[df_unique["cluster_label"] == c_name]
    cust = np.stack((sub["moduleCode"], sub["title"], sub["level"], np.where(sub["is_ai_core"], "Môn Trọng tâm AI/ML", "Môn Khoa học Máy tính Tiêu chuẩn")), axis=-1)
    fig_clusters.add_trace(go.Scatter(
        x=sub["pca_1"],
        y=sub["pca_2"],
        mode="markers",
        name=c_name,
        marker=dict(color=colors_list[idx % len(colors_list)], size=7),
        opacity=0.75,
        customdata=cust,
        hovertemplate="<b>%{customdata[1]}</b><br>"
                      "Mã môn học: <b>%{customdata[0]}</b><br>"
                      "Cấp độ: <b>Level %{customdata[2]}</b><br>"
                      "Phân loại: <b>%{customdata[3]}</b><br>"
                      "Nhóm chủ đề: <b>" + c_name + "</b><extra></extra>"
    ))

fig_clusters.update_layout(
    title=dict(
        text="<b>Phân cụm Không giám sát 3 Nhóm Đề tài Học phần dựa trên Mô tả môn học (TF-IDF + PCA 2D)</b><br>"
             "<span style='font-size:12px; color:#718096;'>K-Means (K=3) tự động phân tách rõ rệt các môn AI/Data, Hệ thống phần mềm và Thuật toán lý thuyết.</span>",
        x=0.04, y=0.93
    ),
    plot_bgcolor=COLOR_BG,
    paper_bgcolor=COLOR_BG,
    font=dict(family="Segoe UI, Arial, sans-serif", color=COLOR_DARK),
    xaxis=dict(showgrid=True, gridcolor=COLOR_LIGHT, linecolor=COLOR_MUTED, title="Thành phần Chính 1 (PCA 1)"),
    yaxis=dict(showgrid=True, gridcolor=COLOR_LIGHT, linecolor=COLOR_MUTED, title="Thành phần Chính 2 (PCA 2)"),
    legend=dict(x=0.04, y=0.85, bgcolor="rgba(255,255,255,0.85)", bordercolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=40, t=85, b=50)
)
fig_clusters.show()
